# 3. High-Cardinality Categorical Encoding: Target & Frequency Encoding

This notebook covers:
1. The **High-Cardinality Dilemma**: Why One-Hot Encoding breaks when a column has hundreds of unique values (e.g., Pincodes, Product IDs, Cities).
2. **Frequency (Count) Encoding**: Replacing categories with their frequency or occurrence counts.
3. **Target (Mean) Encoding**: Replacing categories with the expected mean of the target variable.
4. **Target Leakage & Overfitting**: How modern Scikit-Learn's `TargetEncoder` prevents leakage using built-in **Out-of-Fold (OOF)** cross-validation and smoothing.
5. End-to-end integration using `ColumnTransformer`.

In [1]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import TargetEncoder

# Sample dataset with high-cardinality nominal features and a target
data = {
    'Age': [25, 34, 45, 22, 28, 52, 40, 29, 36, 48],
    'City_Pincode': [
        '500001', '560001', '400001', '500001', '560001', 
        '110001', '500001', '400001', '560001', '600001'
    ],
    'Vehicle_Type': [
        'Sedan', 'SUV', 'Hatchback', 'Sedan', 'SUV', 
        'Sedan', 'Hatchback', 'Sedan', 'SUV', 'Hatchback'
    ],
    'Claim_Fraud': [0, 1, 0, 0, 1, 0, 0, 0, 1, 0]  # Binary Target (y)
}

df = pd.DataFrame(data)
print("=== 1. RAW DATASET (HIGH-CARDINALITY CATEGORIES) ===")
display(df)

=== 1. RAW DATASET (HIGH-CARDINALITY CATEGORIES) ===


,Age,City_Pincode,Vehicle_Type,Claim_Fraud
0,25,500001,Sedan,0
1,34,560001,SUV,1
2,45,400001,Hatchback,0
3,22,500001,Sedan,0
4,28,560001,SUV,1
5,52,110001,Sedan,0
6,40,500001,Hatchback,0
7,29,400001,Sedan,0
8,36,560001,SUV,1
9,48,600001,Hatchback,0


---
## Part 1: Why Not One-Hot Encoding for High Cardinality?

If a feature like `City_Pincode` has **500 unique values**, One-Hot Encoding creates **500 new sparse columns**:
* **Curse of Dimensionality:** Explodes memory usage and slows training.
* **Tree Degradation:** Tree models (Random Forest, XGBoost) struggle because the information is fragmented across hundreds of mostly zero columns.

---
## Part 2: Frequency (Count) Encoding

Frequency encoding replaces each category with the **fraction of times (or raw count)** it appears in the dataset.

$$\text{Frequency}(\text{Category}) = \frac{\text{Count of Category}}{\text{Total Rows}}$$

* **Pros:** Creates zero additional columns, retains popularity/density signal.
* **Cons:** Different categories with identical counts will receive the exact same encoded value.

In [4]:
# Create an explicit copy
df_freq = df.copy()

# Compute frequency (normalized count) of each pincode
pincode_freq_map = df_freq['City_Pincode'].value_counts(normalize=True).to_dict()

# Map frequencies directly onto the column
df_freq['City_Pincode_Freq'] = df_freq['City_Pincode'].map(pincode_freq_map)

# Replace the original column and display full dataset
df_freq_final = df_freq.drop(columns=['City_Pincode'])

print("=== 2. FULL DATASET WITH FREQUENCY ENCODING ===")
display(df_freq_final)
display(df_freq_final.dtypes)

=== 2. FULL DATASET WITH FREQUENCY ENCODING ===


,Age,Vehicle_Type,Claim_Fraud,City_Pincode_Freq
0,25,Sedan,0,0.3
1,34,SUV,1,0.3
2,45,Hatchback,0,0.2
3,22,Sedan,0,0.3
4,28,SUV,1,0.3
5,52,Sedan,0,0.1
6,40,Hatchback,0,0.3
7,29,Sedan,0,0.2
8,36,SUV,1,0.3
9,48,Hatchback,0,0.1


Age                    int64
Vehicle_Type          object
Claim_Fraud            int64
City_Pincode_Freq    float64
dtype: object

---
## Part 3: Target (Mean) Encoding & The Leakage Problem

Target encoding replaces each category with the **average target value** for that group.

$$\text{Target\_Value}(\text{Category } k) = \frac{\sum y \text{ for Category } k}{\text{Count of Category } k}$$

### The Critical Danger: Target Leakage
If you compute the simple mean of the target using the entire dataset:
* Rare categories (e.g., a pincode appearing only once with $y=1$) will get encoded as exactly $1.0$.
* The model memorizes this direct target information during training, leading to **severe overfitting** and failure on test data.

### The Production Fix: Smooth Target Encoding (Scikit-Learn)
Modern Scikit-Learn provides `sklearn.preprocessing.TargetEncoder`:
1. It uses internal **Cross-Validation (Out-of-Fold)** during `.fit_transform()` so a row never uses its own target value to compute its encoding.
2. It applies **Empirical Bayes Smoothing**—shrinking rare categories closer to the overall global mean.

In [9]:
# 1. Separate features and target
X = df.drop(columns=['Claim_Fraud']).copy()
y = df['Claim_Fraud'].copy()

# 2. Build ColumnTransformer with TargetEncoder
# cv=5: internal cross-validation to prevent leakage
# smooth='auto': shrinks small categories toward global mean
ct_target = ColumnTransformer(
    transformers=[
        ('pincode_target_enc', TargetEncoder(cv=5, smooth='auto', random_state=42), ['City_Pincode'])
    ],
    remainder='passthrough',
    verbose_feature_names_out=False
).set_output(transform='pandas')

# 3. Fit and transform features using target y
X_target_encoded = ct_target.fit_transform(X, y)

# 4. Attach target back to display the complete dataset
df_target_final = X_target_encoded.copy()
df_target_final['Claim_Fraud'] = y

print("=== 3. FULL DATASET WITH LEAK-FREE TARGET ENCODING ===")
display(df_target_final)

=== 3. FULL DATASET WITH LEAK-FREE TARGET ENCODING ===


d:\machine_learning_code\.venv\Lib\site-packages\sklearn\model_selection\_split.py:813: UserWarning: The least populated class in y has only 3 members, which is less than n_splits=5.
  warnings.warn(


,City_Pincode,Age,Vehicle_Type,Claim_Fraud
0,0.000,25,Sedan,0
1,1.000,34,SUV,1
2,0.000,45,Hatchback,0
3,0.000,22,Sedan,0
4,1.000,28,SUV,1
5,0.375,52,Sedan,0
6,0.000,40,Hatchback,0
7,0.000,29,Sedan,0
8,1.000,36,SUV,1
9,0.250,48,Hatchback,0
